# Augmented-Covariance Riemannian Decoder

**STOPPED COMPLETED CONFIGURATION.** Full50 reached 51.75% versus 52.45% zero-lag. Do not rerun or tune this exact configuration; workspace-root `AGENTS.md` section 2e is authoritative. Execution fails closed unless `allow_closed_rerun` is explicitly overridden after a new prespecification.

# 1. Setup

In [ ]:
import hashlib, json, platform, sys
from datetime import datetime
from pathlib import Path
WORKING_DIR = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / 'src' / 'liu2024').is_dir())
sys.path.insert(0, str(WORKING_DIR / 'src' / 'liu2024'))
from liu2024_candidate_runner import run_candidate_experiment
print(f'Python: {sys.version.split()[0]} | Platform: {platform.platform()} | Root: {WORKING_DIR}')

# 2. Configuration
## 2.1 CONFIG

In [ ]:
CONFIG = {
    'allow_closed_rerun': False,
    # Paths / run identity
    'artifact_dir': str(WORKING_DIR / 'artifacts' / 'liu2024-augmented-covariance-riemann'),
    'resume_run_dir': None,
    'source_extract_dir': str(WORKING_DIR / 'liu2024_data' / 'liu2024_figshare' / 'sourcedata'),
    'split_manifest_path': str(WORKING_DIR / 'artifacts' / 'liu2024-compact-mi-models' / '20260712_165746_790145_fd8ab986' / 'splits.json'),
    'experiment_name': 'augmented_covariance_riemann',
    'candidate_family': 'augmented_covariance_riemann',
    'config_note': 'Fixed p=2, 62.5 ms delay versus matched zero-lag covariance.',
    # Dataset / preprocessing
    'subjects_to_use': None,
    'target_sfreq': 128,
    'mi_window_seconds': 4.0,
    'average_reference': True,
    'bandpass_hz': [8.0, 30.0],
    # Representation (frozen before outer evaluation)
    'embedding_order': 2,
    'lag_samples': 8,
    'trace_normalize': True,
    # Evaluation / reproducibility
    'seed': 2026,
}


## 2.2 Artifact Creation and Reproducibility

In [ ]:
if not CONFIG.get('allow_closed_rerun', False):
    raise RuntimeError('CLOSED: augmented covariance Full50 is complete and negative; see AGENTS.md section 2e.')
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + hashlib.md5(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:8]
ARTIFACT_DIR = Path(CONFIG['resume_run_dir']).resolve() if CONFIG.get('resume_run_dir') else Path(CONFIG['artifact_dir']) / RUN_ID
if CONFIG.get('resume_run_dir'):
    if not ARTIFACT_DIR.is_dir(): raise FileNotFoundError(f'Resume directory does not exist: {ARTIFACT_DIR}')
    RUN_ID = ARTIFACT_DIR.name
else:
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
(ARTIFACT_DIR / 'config.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print(f'Run ID:     {RUN_ID}')
print(f'Artifacts:  {ARTIFACT_DIR}')

# 3. Load and Prepare Data
Strict marker validation and independent full-trial filtering are executed by `liu2024_prelocal_clean.py`.
# 4. Model
OAS covariance, fold-local Riemannian tangent reference, fold-local scaling, shrinkage LDA.
# 5. Training
Every transform is fitted on the 32 outer-training trials only; fold shards provide resume support.
# 6. Results

In [ ]:
try:
    RUN_METADATA = run_candidate_experiment(CONFIG, ARTIFACT_DIR)
    (ARTIFACT_DIR / 'run.log').write_text('completed\n', encoding='utf-8')
except Exception as error:
    (ARTIFACT_DIR / 'run.log').write_text(f'failed: {type(error).__name__}: {error}\n', encoding='utf-8')
    raise
print(json.dumps(RUN_METADATA['global_metrics'], indent=2))
print(f'\nAll artifacts in: {ARTIFACT_DIR}')